In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# Load data
train_df = pd.read_csv("../data/dataset/processed/go_emotion_dutch.csv")
test_df = pd.read_csv("../data/group_4_url_1_transcript.csv")

# Prepare text
train_texts = train_df['Sentence'].astype(str).tolist()
test_texts = test_df['Sentence'].astype(str).tolist()

# Tokenize
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts + test_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

MAX_LENGTH = 30
train_padded = pad_sequences(train_sequences, maxlen=MAX_LENGTH, padding='post')
test_padded = pad_sequences(test_sequences, maxlen=MAX_LENGTH, padding='post')

# Labels
label_encoder = LabelEncoder()
label_encoder.fit(train_df['Emotion_core'].tolist() + test_df['Emotion_core'].tolist())

train_labels = label_encoder.transform(train_df['Emotion_core'].tolist())
test_labels = label_encoder.transform(test_df['Emotion_core'].tolist())

num_classes = len(label_encoder.classes_)
train_categorical = to_categorical(train_labels, num_classes)
test_categorical = to_categorical(test_labels, num_classes)

# Build SimpleRNN model
model = Sequential([
    Embedding(len(tokenizer.word_index) + 1, 64, input_length=MAX_LENGTH),
    SimpleRNN(32),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.build(input_shape=(None, MAX_LENGTH))

print("RNN Model:")
model.summary()

# Train
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

history = model.fit(
    train_padded, train_categorical,
    validation_data=(test_padded, test_categorical),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# Evaluate
predictions = model.predict(test_padded)
predicted_classes = np.argmax(predictions, axis=1)

accuracy = accuracy_score(test_labels, predicted_classes)
f1 = f1_score(test_labels, predicted_classes, average='weighted')

print(f"\nSimpleRNN Results:")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(test_labels, predicted_classes, target_names=label_encoder.classes_))

SimpleRNN Model:


/Users/florislokhorst/Desktop/School/Y2_A/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 30, 64)         │     2,591,488 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,594,823 (9.90 MB)

 Trainable params: 2,594,823 (9.90 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.3869 - loss: 1.5571 - val_accuracy: 0.1895 - val_loss: 1.6428
Epoch 2/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.4174 - loss: 1.4757 - val_accuracy: 0.4457 - val_loss: 1.5377
Epoch 3/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.4809 - loss: 1.3724 - val_accuracy: 0.4333 - val_loss: 1.5416
Epoch 4/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.5283 - loss: 1.2895 - val_accuracy: 0.4667 - val_loss: 1.5080
Epoch 5/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.5658 - loss: 1.2091 - val_accuracy: 0.4724 - val_loss: 1.5445
Epoch 6/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.6049 - loss: 1.1255 - val_accuracy: 0.4419 - val_loss: 1.6013
Epoch 7/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.6415 - loss: 1.0488 - val_accuracy: 0.4181 - val_loss: 1.6540
Epoch 8/50
1785/1785 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6703 - loss: 0.9783 - 

/Users/florislokhorst/Desktop/School/Y2_A/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/florislokhorst/Desktop/School/Y2_A/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/florislokhorst/Desktop/School/Y2_A/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 